# الدرس 11 - بروتوكول من وكيل إلى وكيل (A2A)


## الإعداد


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ما هو بروتوكول A2A؟

بروتوكول **الوكيل إلى الوكيل (A2A)** هو معيار مفتوح يتيح لوكلاء الذكاء الاصطناعي التواصل،
اكتشاف بعضهم البعض، والتعاون ـ حتى عندما يكونون مبنيين على أُطُر عمل مختلفة أو مستضافين
من قبل خدمات مختلفة.

المفاهيم الرئيسية:

- **الاكتشاف** – يقوم الوكلاء بنشر *بطاقة الوكيل* التي تصف قدراتهم، مما يجعل من
  السهل على الوكلاء الآخرين (أو منسقي العمليات) العثور على المختص المناسب لمهمة ما.
- **تمرير الرسائل** – يتبادل الوكلاء رسائل منظمة من خلال بروتوكول مشترك، بحيث يمكن
  فهم طلب من وكيل ما وتنفيذه من قبل وكيل آخر بغض النظر عن التنفيذ الداخلي له.

- **دورة حياة المهمة** – يحدد A2A حالات مثل *مُقدَّم*، *جارٍ العمل*، *مكتمل*، و
  *فشل*، مما يمنح منسق العمليات رؤية كاملة لكيفية تقدم المهمة المفوضة.

في هذا الدرس نحاكي التعاون بأسلوب A2A عن طريق ربط ثلاثة وكلاء سفر متخصصين
في سير عمل حيث يساهم كل وكيل بخبرته ويمرر النتائج إلى التالي.


## إنشاء وكلاء سفر متخصصين


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## التعاون بين وكلاء متعددين عبر سير العمل

نقوم بربط الوكلاء الثلاثة في سير عمل متسلسل يعكس تمرير الرسائل بين الوكلاء A2A:

1. **CurrencyExchangeAgent** يستقبل طلب المستخدم وينتج إرشادات العملة.
2. **ActivityPlannerAgent** يستقبل السياق المعزز ويضيف توصيات الأنشطة.
3. **TravelManagerAgent** يدمج المدخلين في موجز سفر نهائي.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## فهم A2A في بيئة الإنتاج

في بيئة الإنتاج يفتح بروتوكول A2A سيناريوهات قوية عبر الخدمات:

| القدرة | الوصف |
|---|---|
| **التشغيل البيني بين الأُطُر** | يمكن لوكيل مبني بإطار عمل واحد تفويض المهام لوكيل مبني بأي إطار عمل آخر متوافق مع A2A، مما يتيح تشغيلًا بين المؤسسات فعليًا. |
| **حدود الخدمة** | يمكن للوكلاء العيش في خدمات مصغرة منفصلة، أو مناطق سحابية مختلفة، أو حتى مؤسسات مختلفة مع التعاون بسلاسة. |
| **الاكتشاف الديناميكي** | يمكن لمنسق الاستعلام من سجل بطاقة الوكيل أثناء التشغيل للعثور على المتخصص الأنسب لمهمة فرعية معينة. |
| **البث والإشعارات الفورية** | يدعم A2A أحداث خادم مرسلة (SSE) لتحديثات التقدم في الوقت الفعلي والإشعارات الفورية للمهام طويلة الأمد. |

سير العمل الذي بنيناه أعلاه هو نسخة مبسطة داخل العملية من هذا النمط. في بيئة نشر حقيقية
سيكشف كل وكيل عن نقطة نهاية HTTP، وينشر بطاقة وكيل، ويتواصل
عبر بروتوكول A2A JSON-RPC.


## الملخص

في هذا الدرس تعلمت:

1. **ما هو بروتوكول A2A** — معيار مفتوح لاكتشاف الرسائل بين الوكلاء،
   وإدارة المهام.
2. **كيفية إنشاء وكلاء متخصصين** — وكيل لتبادل العملات، وكيل مخطط الأنشطة،
   ومنسق مدير السفر.
3. **كيفية توصيل الوكلاء في سير عمل** — باستخدام `WorkflowBuilder` لنمذجة التتابع
   في تمرير الرسائل بين الوكلاء.
4. **كيفية عمل A2A في الإنتاج** — تمكين التعاون عبر الأُطر والأنظمة
   مع الاكتشاف الديناميكي والتحديثات المباشرة.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
